# Install Package

In [13]:
#!pip install yfinance pandas matplotlib openbb vectorbt "nbformat>=5.10"

# Trading system development

## Data preparation

In [14]:
import io
import os
import zipfile
from datetime import timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import vectorbt as vbt

from openbb import obb

obb.user.preferences.output_type = "dataframe"

# --- Crypto source: Binance public data dumps (data.binance.vision) ---
# OpenBB/yfinance caps intraday history (1h ~730 days, 1m ~8 days), but
# Binance serves full kline archives (1m back to 2017) as zip files —
# no API key, no rate limit. Same source the DGT paper used.
# Note: timestamps switched from milliseconds to microseconds in 2025 files.
_BINANCE_VISION = "https://data.binance.vision/data/spot/{span}/klines/{symbol}/{interval}"
_KLINE_COLS = ["open_time", "open", "high", "low", "close", "volume", "close_time",
               "quote_volume", "trades", "taker_base_vol", "taker_quote_vol", "ignore"]


def _fetch_kline_zip(url: str, cache_dir: str) -> pd.DataFrame | None:
    """Download (or reuse cached) kline zip; None if the file doesn't exist."""
    cache_path = os.path.join(cache_dir, url.rsplit("/", 1)[-1])
    if os.path.exists(cache_path):
        raw = open(cache_path, "rb").read()
    else:
        r = requests.get(url, timeout=120)
        if r.status_code == 404:
            return None
        r.raise_for_status()
        raw = r.content
        with open(cache_path, "wb") as f:
            f.write(raw)
    zf = zipfile.ZipFile(io.BytesIO(raw))
    df = pd.read_csv(zf.open(zf.namelist()[0]), header=None, names=_KLINE_COLS)
    df = df[pd.to_numeric(df["open_time"], errors="coerce").notna()]  # drop header row if present
    return df


def load_binance_klines(symbol: str = "BTCUSDT", interval: str = "1m",
                        start: str = "2025-01-01", end: str | None = None,
                        cache_dir: str = "binance_cache") -> pd.DataFrame:
    """
    Load OHLCV klines from data.binance.vision bulk archives.

    Complete months come from monthly zips, the current month from daily
    zips (published with ~1 day lag). Zips are cached in `cache_dir` so
    reruns don't re-download.

    Parameters
    ----------
    symbol : str
        Binance spot pair, e.g. "BTCUSDT", "ETHUSDT".
    interval : str
        Kline interval: "1m", "5m", "15m", "1h", "4h", "1d", "1w", "1M".
    start, end : str
        "YYYY-MM-DD"; end=None means up to the latest published file.
    """
    os.makedirs(cache_dir, exist_ok=True)
    now = pd.Timestamp.now("UTC").tz_localize(None)
    start_ts = pd.Timestamp(start)
    end_ts = pd.Timestamp(end) if end else now

    frames = []
    last_full_month = now.to_period("M") - 1

    # Monthly archives for every complete month in range
    month = start_ts.to_period("M")
    while month <= min(end_ts.to_period("M"), last_full_month):
        url = (_BINANCE_VISION.format(span="monthly", symbol=symbol, interval=interval)
               + f"/{symbol}-{interval}-{month}.zip")
        df = _fetch_kline_zip(url, cache_dir)
        if df is not None:
            frames.append(df)
        month += 1

    # Daily archives for the current (incomplete) month
    if end_ts.to_period("M") > last_full_month:
        day = max(start_ts, (last_full_month + 1).to_timestamp()).normalize()
        last_day = min(end_ts, now - timedelta(days=1)).normalize()
        while day <= last_day:
            url = (_BINANCE_VISION.format(span="daily", symbol=symbol, interval=interval)
                   + f"/{symbol}-{interval}-{day:%Y-%m-%d}.zip")
            df = _fetch_kline_zip(url, cache_dir)
            if df is not None:
                frames.append(df)
            day += timedelta(days=1)

    if not frames:
        raise ValueError(f"No data found for {symbol} {interval} in {start} .. {end}")

    out = pd.concat(frames, ignore_index=True)
    ts = pd.to_numeric(out["open_time"])
    unit = "us" if ts.iloc[0] > 1e14 else "ms"   # microseconds since 2025 files
    out.index = pd.to_datetime(ts, unit=unit)
    out.index.name = "date"
    out = out[["open", "high", "low", "close", "volume"]].astype(float)
    out = out.sort_index()
    return out.loc[(out.index >= start_ts) & (out.index <= end_ts)]


def load_data(sector: str, symbol: str, start_date: str, end_date: str, interval: str = "1d") -> pd.DataFrame:
    """
    Load OHLCV price data.

    Crypto comes from Binance public archives (full intraday history);
    stocks and forex come from OpenBB (obb).

    Parameters
    ----------
    sector : str
        One of "stock", "crypto", "forex".
    symbol : str
        Ticker symbol. Stock e.g. "AAPL". Crypto e.g. "BTCUSD" or "BTCUSDT"
        (a USD suffix is mapped to Binance's USDT pair). Forex e.g. "EURUSD".
    start_date, end_date : str
        Format "YYYY-MM-DD".
    interval : str
        Price timeframe, e.g. "1m", "5m", "15m", "1h", "1d", "1W", "1M".
    """
    sector = sector.lower()

    if sector == "crypto":
        pair = symbol.upper().replace("-", "").replace("/", "")
        if pair.endswith("USD"):
            pair += "T"                            # BTCUSD -> BTCUSDT
        # Binance interval names are lowercase, except monthly "1M"
        b_interval = interval if interval == "1M" else interval.lower()
        return load_binance_klines(pair, b_interval, start=start_date, end=end_date)

    if sector == "stock":
        df = obb.equity.price.historical(
            symbol=symbol, start_date=start_date, end_date=end_date, interval=interval
        )
    elif sector == "forex":
        df = obb.currency.price.historical(
            symbol=symbol, start_date=start_date, end_date=end_date, interval=interval
        )
    else:
        raise ValueError(f"Unsupported sector: {sector}. Choose 'stock', 'crypto', or 'forex'.")

    df.columns = [c.lower() for c in df.columns]
    return df


In [15]:
# Example usage
SECTOR = "crypto"       # "stock" | "crypto" | "forex"
SYMBOL = "BTCUSD"
START_DATE = "2020-01-01"
END_DATE = "2021-01-01"
INTERVAL = "15m"

data = load_data(SECTOR, SYMBOL, START_DATE, END_DATE, INTERVAL)
data.head()


,open,high,low,close,volume
date,,,,,
2020-01-01 00:00:00,7195.24,7196.25,7178.20,7180.97,202.942868
2020-01-01 00:15:00,7180.97,7186.40,7175.47,7178.45,128.242654
2020-01-01 00:30:00,7178.19,7185.44,7176.23,7179.56,83.487458
2020-01-01 00:45:00,7179.35,7183.98,7175.46,7177.02,97.141921
2020-01-01 01:00:00,7176.47,7194.04,7175.71,7190.86,103.520522


## Data Analysis & Signal Process

### Stats Indicator

In [16]:
# =============================================================================
# Stats Indicator engine
# -----------------------------------------------------------------------------
# add_indicators(data, specs) appends indicator columns to an OHLCV DataFrame
# (expects lowercase columns: open, high, low, close, volume).
#
# `specs` is a list where each item is either:
#     "rsi"                                  -> use default parameters
#     ("rsi", {"period": 14, "smooth": 3})   -> configure parameters
#
# Supported indicators (name -> parameters, defaults shown):
#     sma         period=20, source="close"
#     ema         period=20, source="close"
#     rsi         period=14, smooth=None        (smooth=3 -> extra SMA on RSI)
#     stoch       k_period=14, d_period=3
#     macd        fast=12, slow=26, signal=9
#     bbands      period=20, n_std=2.0
#     atr         period=14
#     roc         period=10                     (rate of change, %)
#     zscore      period=20, source="close"
#     obv         (no parameters)
#     volatility  window=30, annualize=None     (rolling std of log returns;
#                                                annualize=365*24*4 for 15m crypto)
#     vol_regime  window=30, lookback=500, q_low=0.33, q_high=0.66
#                 -> 0 = low vol, 1 = mid, 2 = high (vs rolling history quantiles)
# =============================================================================


def _rsi(close: pd.Series, period: int = 14) -> pd.Series:
    """Wilder's RSI."""
    delta = close.diff()
    gain = delta.clip(lower=0.0)
    loss = -delta.clip(upper=0.0)
    avg_gain = gain.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100.0 - 100.0 / (1.0 + rs)


def _true_range(df: pd.DataFrame) -> pd.Series:
    prev_close = df["close"].shift(1)
    return pd.concat(
        [
            df["high"] - df["low"],
            (df["high"] - prev_close).abs(),
            (df["low"] - prev_close).abs(),
        ],
        axis=1,
    ).max(axis=1)


# --- indicator builders: each returns {column_name: Series} ------------------

def _ind_sma(df, period=20, source="close"):
    return {f"sma_{period}": df[source].rolling(period).mean()}


def _ind_ema(df, period=20, source="close"):
    return {f"ema_{period}": df[source].ewm(span=period, adjust=False).mean()}


def _ind_rsi(df, period=14, smooth=None):
    rsi = _rsi(df["close"], period)
    out = {f"rsi_{period}": rsi}
    if smooth:  # e.g. RSI(14, 3) -> extra 3-bar SMA smoothing
        out[f"rsi_{period}_{smooth}"] = rsi.rolling(smooth).mean()
    return out


def _ind_stoch(df, k_period=14, d_period=3):
    low_min = df["low"].rolling(k_period).min()
    high_max = df["high"].rolling(k_period).max()
    k = 100.0 * (df["close"] - low_min) / (high_max - low_min)
    return {
        f"stoch_k_{k_period}": k,
        f"stoch_d_{k_period}_{d_period}": k.rolling(d_period).mean(),
    }


def _ind_macd(df, fast=12, slow=26, signal=9):
    ema_fast = df["close"].ewm(span=fast, adjust=False).mean()
    ema_slow = df["close"].ewm(span=slow, adjust=False).mean()
    macd = ema_fast - ema_slow
    sig = macd.ewm(span=signal, adjust=False).mean()
    return {
        f"macd_{fast}_{slow}": macd,
        f"macd_signal_{signal}": sig,
        f"macd_hist_{fast}_{slow}_{signal}": macd - sig,
    }


def _ind_bbands(df, period=20, n_std=2.0):
    mid = df["close"].rolling(period).mean()
    std = df["close"].rolling(period).std()
    upper = mid + n_std * std
    lower = mid - n_std * std
    return {
        f"bb_mid_{period}": mid,
        f"bb_upper_{period}": upper,
        f"bb_lower_{period}": lower,
        f"bb_width_{period}": (upper - lower) / mid,
    }


def _ind_atr(df, period=14):
    tr = _true_range(df)
    atr = tr.ewm(alpha=1.0 / period, min_periods=period, adjust=False).mean()
    return {f"atr_{period}": atr}


def _ind_roc(df, period=10):
    return {f"roc_{period}": df["close"].pct_change(period) * 100.0}


def _ind_zscore(df, period=20, source="close"):
    s = df[source]
    mean = s.rolling(period).mean()
    std = s.rolling(period).std()
    return {f"zscore_{period}": (s - mean) / std}


def _ind_obv(df):
    direction = np.sign(df["close"].diff()).fillna(0.0)
    return {"obv": (direction * df["volume"]).cumsum()}


def _ind_volatility(df, window=30, annualize=None):
    log_ret = np.log(df["close"] / df["close"].shift(1))
    vol = log_ret.rolling(window).std()
    if annualize:
        vol = vol * np.sqrt(annualize)
    return {f"vol_{window}": vol}


def _ind_vol_regime(df, window=30, lookback=500, q_low=0.33, q_high=0.66):
    """
    Volatility regime: realized vol vs its own rolling-history quantiles.
      0 = low vol   (vol <= q_low quantile of past `lookback` bars)
      1 = mid vol
      2 = high vol  (vol >= q_high quantile)
    """
    log_ret = np.log(df["close"] / df["close"].shift(1))
    vol = log_ret.rolling(window).std()
    lo = vol.rolling(lookback, min_periods=window).quantile(q_low)
    hi = vol.rolling(lookback, min_periods=window).quantile(q_high)
    regime = pd.Series(np.nan, index=df.index)
    regime[vol <= lo] = 0
    regime[(vol > lo) & (vol < hi)] = 1
    regime[vol >= hi] = 2
    return {f"vol_{window}": vol, "vol_regime": regime}


INDICATORS = {
    "sma": _ind_sma,
    "ema": _ind_ema,
    "rsi": _ind_rsi,
    "stoch": _ind_stoch,
    "macd": _ind_macd,
    "bbands": _ind_bbands,
    "atr": _ind_atr,
    "roc": _ind_roc,
    "zscore": _ind_zscore,
    "obv": _ind_obv,
    "volatility": _ind_volatility,
    "vol_regime": _ind_vol_regime,
}


def add_indicators(df: pd.DataFrame, specs, inplace: bool = False) -> pd.DataFrame:
    """
    Add indicator columns to an OHLCV DataFrame.

    Parameters
    ----------
    df : DataFrame with lowercase columns open/high/low/close/volume.
    specs : list of indicator specs; each item is "name" or ("name", {params}).
    inplace : if False (default), work on a copy and return it.

    Returns
    -------
    DataFrame with the new indicator columns appended.
    """
    out = df if inplace else df.copy()
    for spec in specs:
        if isinstance(spec, str):
            name, params = spec, {}
        else:
            name, params = spec[0], (spec[1] if len(spec) > 1 else {})
        name = name.lower()
        if name not in INDICATORS:
            raise ValueError(
                f"Unknown indicator '{name}'. Available: {sorted(INDICATORS)}"
            )
        for col, series in INDICATORS[name](out, **params).items():
            out[col] = series
    return out


# --- Example usage -----------------------------------------------------------
data = add_indicators(data, [
    ("vol_regime", {"window": 30, "lookback": 500}),
])
data.tail()

,open,high,low,close,volume,vol_30,vol_regime
date,,,,,,,
2020-12-31 23:00:00,29100.83,29110.35,28925.00,29071.22,350.002996,0.003244,0.0
2020-12-31 23:15:00,29070.85,29090.39,28951.36,29002.88,420.013349,0.003281,0.0
2020-12-31 23:30:00,29002.88,29003.16,28810.24,28824.95,575.964775,0.003389,0.0
2020-12-31 23:45:00,28827.49,28999.00,28780.00,28923.63,630.438179,0.003438,0.0
2021-01-01 00:00:00,28923.63,29017.50,28690.17,28752.80,840.077569,0.003476,0.0


## Create Trading Strategy

## Backtesting